<a href="https://colab.research.google.com/github/iamamnabilal40/Sentiment-Analysis-App-CineSense-/blob/main/Sentiment_Analysis_App_(CineSense).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Objective: Demonstrate a complete, Python application covering data preparation, model training, evaluation, and an interactive deployment layer.

In [2]:
!pip install scikit-learn joblib pandas -q

In [1]:
POSITIVE_REVIEWS = [
    "An absolute masterpiece from start to finish, I was hooked the entire time.",
    "The acting was superb and the story kept me on the edge of my seat.",
    "One of the best films I have seen this year, truly unforgettable.",
    "Beautifully shot with a soundtrack that perfectly matched every scene.",
    "The chemistry between the leads made the romance feel completely real.",
    "A brilliant script paired with flawless direction, I loved every minute.",
    "This movie exceeded all my expectations, I would watch it again immediately.",
    "The plot twists were genius and kept the audience guessing until the end.",
    "A heartwarming story that left me smiling long after the credits rolled.",
    "The visual effects were stunning and elevated the entire viewing experience.",
    "Incredible performances all around, the cast truly brought the story to life.",
    "A gripping thriller that never lets up, I could not look away.",
    "The dialogue was witty and sharp, easily the funniest film of the year.",
    "A touching and emotional journey that resonated with me deeply.",
    "The pacing was perfect, never a dull moment throughout the entire runtime.",
    "This is exactly the kind of movie that reminds you why you love cinema.",
    "The director did an amazing job balancing humor with genuine emotion.",
    "A stunning debut, I cannot wait to see what this filmmaker does next.",
    "The soundtrack alone is worth watching this movie for, absolutely beautiful.",
    "A powerful and inspiring story told with heart and confidence.",
    "The cinematography was breathtaking, every frame looked like a painting.",
    "I laughed, I cried, and I left the theater completely satisfied.",
    "A wonderfully crafted film with a message that stays with you.",
    "The performances felt authentic and the story never lost my attention.",
    "Easily one of the most enjoyable movies I have watched in years.",
    "A clever and original concept executed with real skill and care.",
    "The ending was so satisfying, it tied everything together perfectly.",
    "This film is a triumph of storytelling and technical craftsmanship.",
    "Every character felt fully realized and the world building was excellent.",
    "A must watch for anyone who appreciates thoughtful, well made cinema.",
    "The energy of this movie is infectious, I was smiling the whole way through.",
    "A brilliant blend of action and emotion that never feels forced.",
    "The screenplay is tight, smart, and full of memorable moments.",
    "This movie restored my faith in original, creative filmmaking.",
    "An emotionally rich film that treats its audience with real intelligence.",
    "The lead performance was career defining, truly award worthy work.",
    "A joyful, uplifting film that is impossible not to love.",
    "The tension builds beautifully and pays off in a satisfying finale.",
    "A near perfect film with barely a single wasted scene.",
    "I was completely immersed in this world from the very first minute.",
]

NEGATIVE_REVIEWS = [
    "The plot was predictable and the characters felt completely flat.",
    "I found myself checking the time throughout this painfully slow movie.",
    "The dialogue was cringeworthy and the acting felt forced and unnatural.",
    "A complete waste of time, the story made no sense whatsoever.",
    "The pacing dragged so much that I almost fell asleep halfway through.",
    "None of the jokes landed and the humor felt awkward and outdated.",
    "The special effects looked cheap and took me out of the experience.",
    "A disappointing sequel that fails to live up to the original film.",
    "The ending felt rushed and left way too many plot holes unresolved.",
    "I could not connect with any of the characters in this movie.",
    "The script was weak and the performances did nothing to save it.",
    "This film tries too hard and ends up feeling shallow and hollow.",
    "The story dragged on far longer than it needed to, I was bored.",
    "A forgettable movie with clichéd writing and lazy direction.",
    "The soundtrack was distracting and clashed with the tone of the film.",
    "I regret spending money on a ticket for this poorly made movie.",
    "The chemistry between the leads was nonexistent, it felt awkward.",
    "This movie relies on cheap jump scares instead of real tension.",
    "The editing was choppy and made the story hard to follow at times.",
    "A bland and uninspired film that adds nothing new to the genre.",
    "The acting was wooden and the emotional scenes fell completely flat.",
    "I found the plot confusing and full of unnecessary subplots.",
    "This film is overly long and could have been cut by an hour.",
    "The humor felt forced and none of the punchlines were funny.",
    "A frustrating watch with pacing issues from start to finish.",
    "The characters made decisions that made no logical sense at all.",
    "This movie is a shallow cash grab with no real substance.",
    "The dialogue felt stiff and unnatural throughout the entire film.",
    "I was disappointed by how predictable every plot twist turned out to be.",
    "The film lacked any real emotional depth or meaningful character growth.",
    "A messy script with an unclear tone that never finds its footing.",
    "The performances felt phoned in and lacked any genuine energy.",
    "This is one of the weakest films I have seen from this director.",
    "The story felt recycled from countless other movies in the genre.",
    "A tedious experience with a runtime that felt twice as long as it was.",
    "The villain was poorly written and never felt like a real threat.",
    "I left the theater feeling let down by the underwhelming finale.",
    "The movie relies on style over substance and it shows badly.",
    "A confusing narrative structure that undermines the entire story.",
    "This film simply fails to deliver on its promising premise.",
]


def _generate_templated_reviews():
    """Augments hand-written reviews with template-generated ones so the
    model has enough examples per class to learn robust patterns."""
    subjects = [
        "the movie", "this film", "the story", "the plot", "the screenplay",
        "the direction", "this picture", "the production", "the movie overall",
        "this entire film",
    ]
    positive_adjs = [
        "brilliant", "outstanding", "captivating", "delightful", "masterful",
        "excellent", "phenomenal", "engaging", "impressive", "remarkable",
        "superb", "wonderful", "fantastic", "compelling", "inspired",
    ]
    negative_adjs = [
        "boring", "dull", "disappointing", "forgettable", "tedious",
        "poorly written", "lifeless", "underwhelming", "bad", "awful", "terrible",
        "unoriginal", "predictable", "pointless", "confusing", "shallow"
    ]

    generated_positive_reviews = []
    generated_negative_reviews = []

    for subj in subjects:
        for adj in positive_adjs:
            generated_positive_reviews.append(f"{subj} was {adj}.")
        for adj in negative_adjs:
            generated_negative_reviews.append(f"{subj} was {adj}.")

    return generated_positive_reviews, generated_negative_reviews

# Call the function to generate templated reviews and extend the original lists
generated_pos, generated_neg = _generate_templated_reviews()
POSITIVE_REVIEWS.extend(generated_pos)
NEGATIVE_REVIEWS.extend(generated_neg)

# Combine all reviews and create labels
texts = POSITIVE_REVIEWS + NEGATIVE_REVIEWS
labels = [1] * len(POSITIVE_REVIEWS) + [0] * len(NEGATIVE_REVIEWS)


In [3]:
import json
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000)),
])

param_grid = {
    "tfidf__max_features": [500, 1000, None],
    "tfidf__min_df": [1, 2],
    "clf__C": [0.1, 1, 5, 10],
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

metrics = {
    "best_params": grid_search.best_params_,
    "accuracy": round(accuracy_score(y_test, y_pred), 4),
    "precision": round(precision_score(y_test, y_pred), 4),
    "recall": round(recall_score(y_test, y_pred), 4),
    "f1_score": round(f1_score(y_test, y_pred), 4),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
}

# refit on full dataset for deployment
best_model.fit(texts, labels)
joblib.dump(best_model, "sentiment_model.joblib")

print(json.dumps(metrics, indent=2))

{
  "best_params": {
    "clf__C": 5,
    "tfidf__max_features": 500,
    "tfidf__min_df": 2
  },
  "accuracy": 0.9231,
  "precision": 0.9,
  "recall": 0.9474,
  "f1_score": 0.9231,
  "confusion_matrix": [
    [
      36,
      4
    ],
    [
      2,
      36
    ]
  ]
}


In [4]:
def predict_sentiment(text):
    pred = best_model.predict([text])[0]
    proba = best_model.predict_proba([text])[0]
    label = "Positive 😊" if pred == 1 else "Negative 😞"
    print(f"Review: {text}")
    print(f"Sentiment: {label} (confidence: {proba[pred]:.1%})\n")

predict_sentiment("The movie was a stunning achievement with brilliant performances.")
predict_sentiment("Painfully slow and predictable, I nearly fell asleep.")

# try your own:
predict_sentiment("Type your own review here")

Review: The movie was a stunning achievement with brilliant performances.
Sentiment: Positive 😊 (confidence: 94.1%)

Review: Painfully slow and predictable, I nearly fell asleep.
Sentiment: Negative 😞 (confidence: 94.4%)

Review: Type your own review here
Sentiment: Positive 😊 (confidence: 52.6%)



In [5]:
while True:
    text = input("Enter a movie review (or 'quit' to stop): ")
    if text.lower() == "quit":
        break
    predict_sentiment(text)

Enter a movie review (or 'quit' to stop): movie is bad. 
Review: movie is bad. 
Sentiment: Negative 😞 (confidence: 93.6%)

Enter a movie review (or 'quit' to stop): movie is good.
Review: movie is good.
Sentiment: Positive 😊 (confidence: 51.3%)

Enter a movie review (or 'quit' to stop): movie is just so so.
Review: movie is just so so.
Sentiment: Positive 😊 (confidence: 51.3%)

Enter a movie review (or 'quit' to stop): movie is intresting.
Review: movie is intresting.
Sentiment: Positive 😊 (confidence: 51.3%)

Enter a movie review (or 'quit' to stop): Movie is terrible
Review: Movie is terrible
Sentiment: Negative 😞 (confidence: 93.6%)

Enter a movie review (or 'quit' to stop): Movie is ao cheap . The graphics are tarash.
Review: Movie is ao cheap . The graphics are tarash.
Sentiment: Negative 😞 (confidence: 75.9%)

Enter a movie review (or 'quit' to stop): quit


Pipeline:

  1. `dataset.py` — curated + template-augmented labeled movie reviews
  2. `train_model.py` — TF-IDF vectorization, Logistic Regression,
   grid search hyperparameter tuning, evaluation, model persistence
  3. `app.py` — Streamlit interface for live predictions and metrics

Tech stack:

 Python, scikit-learn, pandas, joblib, Streamlit

Possible extensions:

 - Swap in a larger dataset (e.g. IMDB 50k reviews)
 - Replace TF-IDF + Logistic Regression with a fine tuned BERT model
 - Add batch CSV upload for bulk sentiment scoring
 - Deploy to Streamlit Community Cloud for a public link